# Признаки и обучение на всех получасовых стартах
Каждая модель обучается на всех стартах с известной меткой своего горизонта. Подбор, локальный тест и сабмит используют старт в 10:30.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA = Path("data")
TRAIN_FILE = "train_solo_track.parquet"
TEST_FILE = "test_solo_track.parquet"
if not (DATA / TRAIN_FILE).is_file() and Path("/kaggle/input").exists():
    matches = [p.parent for p in Path("/kaggle/input").rglob(TRAIN_FILE)
               if (p.parent / TEST_FILE).is_file()]
    if len(matches) != 1:
        raise ValueError(f"Укажите DATA вручную. Найденные папки: {matches}")
    DATA = matches[0]
OUT = Path("artifacts/features")
OUT.mkdir(parents=True, exist_ok=True)
TARGET = "target_1h"
STATUSES = [f"status_{i}" for i in range(1, 7)]
STEP = pd.Timedelta(minutes=30)

## Получасовая сетка
Проверяем полноту панели перед использованием позиционных лагов. Строки массивов - моменты времени, столбцы - маршруты.


In [ ]:
history = pd.read_parquet(DATA / TRAIN_FILE).sort_values(["timestamp", "route_id"])
test = pd.read_parquet(DATA / TEST_FILE)
times = pd.DatetimeIndex(history.timestamp.unique())
routes = np.sort(history.route_id.unique())
if not (history.notna().all().all() and test.notna().all().all()):
    raise ValueError("В данных есть пропуски")
if not (not history.duplicated(["route_id", "timestamp"]).any()):
    raise ValueError("Повторяется ключ маршрута и времени")
if not (times.equals(pd.date_range(times[0], times[-1], freq=STEP))):
    raise ValueError("Нарушена получасовая сетка")
if not (len(history) == len(times) * len(routes)):
    raise ValueError("Неполная панель маршрутов")
if not (test.id.is_unique and not test.duplicated(["route_id", "timestamp"]).any()):
    raise ValueError("Повторы идентификаторов или ключей теста")
end = times[-1]
if not ((end.hour, end.minute) == (10, 30)):
    raise ValueError("Последняя точка истории должна быть в 10:30")
expected = pd.MultiIndex.from_product([routes, [end + h*STEP for h in range(1, 9)]],
                                      names=["route_id", "timestamp"])
if not (expected.difference(pd.MultiIndex.from_frame(test[["route_id", "timestamp"]])).empty):
    raise ValueError("Тест не покрывает все ожидаемые маршруты и горизонты")
if not (len(test) == len(expected)):
    raise ValueError("Число строк теста отличается от ожидаемого")
values = {name: history[name].to_numpy().reshape(len(times), len(routes))
          for name in [TARGET, *STATUSES]}
del history

## Исторические признаки
Для каждого получаса используем данные до этого момента включительно. В начале истории лаги без наблюдений остаются NaN, средние считаются по доступным точкам.


In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

target = values[TARGET]
def past(array, positions):
    result = array[np.maximum(positions, 0)].astype(np.float32)
    result[positions < 0] = np.nan
    return result

def average_available(matrices):
    total = np.zeros_like(matrices[0])
    count = np.zeros_like(total)
    for matrix in matrices:
        total += np.nan_to_num(matrix)
        count += np.isfinite(matrix)
    return np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)

def rolling(array, steps):
    return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)

means = {(name, steps): rolling(array, steps) for name, array in values.items() for steps in (4, 12)}
level24 = rolling(target, 48)

def common_features(origins):
    common = {}
    for k in (0, 1, 2, 4):
        common[f"target_back_{k}"] = past(target, origins - k)
    for name, array in values.items():
        for steps in (4, 12):
            common[f"{name}_mean_{steps}"] = means[name, steps][origins]
        if name in STATUSES:
            common[f"{name}_latest"] = array[origins]
            common[f"{name}_change_1h"] = array[origins] - past(array, origins - 2)
    common["target_vs_mean_6h"] = target[origins] - means[TARGET, 12][origins]
    recent = means[TARGET, 4][origins]
    usual = average_available([past(means[TARGET, 4], origins - day*48) for day in range(1, 8)])
    common["morning_level_ratio"] = np.divide(recent, usual, out=np.full_like(recent, np.nan), where=usual > 0)
    return common

## Признаки слота и сохранение
Каждая строка соответствует одному маршруту, дате и горизонту. Метка y доступна только для истории. Колонки id, cutoff, timestamp и y служебные.


In [ ]:
def horizon_dataset(h, origins):
    slot = origins + h
    dates = times[origins] + h * STEP
    frame = pd.DataFrame({
        "cutoff": times[origins].repeat(len(routes)),
        "timestamp": dates.repeat(len(routes)),
        "route_id": np.tile(routes, len(origins)),
    })
    for name, matrix in common_features(origins).items():
        frame[name] = matrix.ravel()
    for name, index, period in [
        ("weekday", dates.dayofweek, 7),
        ("half_hour", dates.hour * 2 + dates.minute // 30 + 1, 48),
    ]:
        phase = index if name == "weekday" else index - 1
        angle = 2 * np.pi * phase / period
        frame[name + "_sin"] = np.repeat(np.sin(angle), len(routes))
        frame[name + "_cos"] = np.repeat(np.cos(angle), len(routes))
    for days in (1, 2, 7):
        frame[f"slot_days_back_{days}"] = past(target, slot - days*48).ravel()
    for name, days in [("slot_daily7", range(1, 8)), ("slot_weekly4", (7, 14, 21, 28))]:
        frame[name] = average_available([past(target, slot - d*48) for d in days]).ravel()
    profile = np.zeros((len(origins), len(routes)))
    counts = np.zeros_like(profile)
    for day in range(1, 29):
        level = past(level24, origins - day*48)
        adjusted = past(target, slot - day*48) * level24[origins] / np.maximum(level, 1)
        profile += np.nan_to_num(adjusted)
        counts += np.isfinite(adjusted)
    frame["profile28"] = np.divide(profile, counts, out=np.full_like(profile, np.nan), where=counts > 0).ravel()
    if h == 1:
        lower = np.zeros_like(target[origins])
        upper = past(target, origins - 2)
        for position in (origins - 1, origins):
            total = past(target, position)
            lower, upper = np.maximum(0, total - upper), np.minimum(total, total - lower)
        seasonal = frame[["slot_daily7", "slot_weekly4"]].mean(axis=1).to_numpy()
        frame["overlap"] = np.where((lower <= upper).ravel(),
                                    (lower + upper).ravel() / 2 + seasonal / 2, seasonal)
    else:
        frame["overlap"] = np.nan
    numeric = frame.columns.difference(["cutoff", "timestamp", "route_id"])
    return frame.astype({name: "float32" for name in numeric})

def write_horizon(h):
    stop = len(times) - h
    path = OUT / f"history_h{h}.parquet"
    temporary = path.with_suffix(".partial")
    writer = None
    try:
        for start in range(0, stop, 48):
            origins = np.arange(start, min(start + 48, stop))
            frame = horizon_dataset(h, origins)
            frame["y"] = target[origins + h].ravel()
            table = pa.Table.from_pandas(frame, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(temporary, table.schema, compression="zstd")
            writer.write_table(table)
    finally:
        if writer is not None:
            writer.close()
    temporary.replace(path)
    features = horizon_dataset(h, np.array([len(times) - 1]))
    requests = test.loc[test.timestamp.eq(end + h*STEP)]
    future = requests.merge(features, on=["route_id", "timestamp"], how="left", validate="one_to_one")
    if not future.cutoff.eq(end).all() or not future.id.equals(requests.id.reset_index(drop=True)):
        raise ValueError("Ошибка сопоставления тестовых строк")
    future.to_parquet(OUT / f"submission_h{h}.parquet", index=False)
    print(f"h={h}: история {stop * len(routes):,}, тест {len(future):,}")

for h in range(1, 9):
    write_horizon(h)

del values, target, means, level24

## Обучение CatBoost на GPU
Последние 14 размеченных дат оставляем для оценки. Предыдущие 14 дат используем для выбора числа деревьев по MAE. Все горизонты используют одинаковые даты. После оценки обучаем каждую модель на всей истории с выбранным числом деревьев.


In [ ]:
from catboost import CatBoostRegressor
from catboost.utils import get_gpu_device_count

if get_gpu_device_count() < 1:
    raise RuntimeError("Для обучения включите GPU в настройках Kaggle Notebook.")
MODEL_DIR = OUT.parent / "models"
MODEL_DIR.mkdir(exist_ok=True)
dates = times[(times.hour == 10) & (times.minute == 30) & (times + 8 * STEP <= end)]
if len(dates) <= 28:
    raise ValueError("Недостаточно дат для временного разбиения")
tune_start, holdout_start = dates[-28], dates[-14]
print("Начало подбора:", tune_start, "Начало отложенной проверки:", holdout_start)

def make_model(trees):
    return CatBoostRegressor(iterations=trees, depth=7, learning_rate=0.05,
                             loss_function="MAE", eval_metric="MAE", task_type="GPU",
                             devices="0", cat_features=["route_id"], random_seed=42,
                             allow_writing_files=False, verbose=100)

def mean_daily_score(frame, prediction):
    errors = np.asarray(prediction) - frame.y.to_numpy()
    totals = pd.DataFrame({"cutoff": frame.cutoff.dt.normalize(), "volume": frame.y,
                           "error": errors, "absolute": np.abs(errors)}).groupby("cutoff").sum()
    return ((totals.absolute + totals.error.abs()) / totals.volume).mean()

def score_row(frame, predictions, split):
    errors = np.asarray(predictions) - frame.y.to_numpy()
    volume = frame.y.sum()
    wape = np.abs(errors).sum() / volume
    bias = np.abs(errors.sum()) / volume
    return {"выборка": split, "даты": frame.cutoff.dt.normalize().nunique(), "строки": len(frame),
            "WAPE": wape, "|Relative Bias|": bias, "метрика конкурса": wape + bias,
            "среднее по дням": mean_daily_score(frame, predictions)}

def variants(h):
    return [("catboost", 1), ("scaled", 100_000)] if h == 2 else [("catboost", 1)]

def model_columns(data):
    return data.columns.drop(["timestamp", "cutoff", "y", "profile28", "overlap"]).tolist()

split_dates = {"train": dates[dates < tune_start],
               "подбор": dates[(dates >= tune_start) & (dates < holdout_start)],
               "test": dates[dates >= holdout_start]}
display(pd.DataFrame([{"выборка": name, "с": period.min(), "по": period.max(), "даты": len(period)}
                      for name, period in split_dates.items()]))

In [ ]:
validation_parts = []
tree_counts = {}
evaluation_tables = {}
for h in range(1, 9):
    data = pd.read_parquet(OUT / f"history_h{h}.parquet")
    columns = model_columns(data)
    train = data.timestamp < tune_start.normalize()
    evaluate = data.cutoff.isin(dates[-28:])
    tune = evaluate & (data.cutoff < holdout_start)
    evaluation = data.loc[evaluate, ["cutoff", "timestamp", "route_id", "y", "profile28", "overlap"]].copy()
    evaluation["horizon"] = h
    evaluation["seasonal"] = data.loc[evaluate, ["slot_daily7", "slot_weekly4"]].mean(axis=1)
    for variant, scale in variants(h):
        model = make_model(1500)
        model.fit(data.loc[train, columns], data.loc[train, "y"] / scale,
                  eval_set=(data.loc[tune, columns], data.loc[tune, "y"] / scale),
                  early_stopping_rounds=100, use_best_model=True)
        tree_counts[h, variant] = model.tree_count_
        evaluation[variant] = (model.predict(data.loc[evaluate, columns]) * scale).clip(min=0)
        test_rows = evaluation.loc[evaluation.cutoff >= holdout_start]
        train_pred = (model.predict(data.loc[train, columns]) * scale).clip(min=0)
        table = pd.DataFrame([score_row(data.loc[train], train_pred, "train"),
                              score_row(test_rows, test_rows[variant], "test")])
        evaluation_tables[h, variant] = table
        print(f"h={h}, {variant}, деревьев: {model.tree_count_}")
        display(table)
        del model
    validation_parts.append(evaluation)
validation = pd.concat(validation_parts, ignore_index=True)
validation.to_parquet(OUT.parent / "validation_predictions.parquet", index=False)

## Смешивание и калибровка
Подбираем общий вес модели и множитель на ранних проверочных датах. Сравниваем исходные и скорректированные прогнозы на последних 14 датах. Для submission используем настройки раннего периода. Оценка каждой даты объединяет все 8 горизонтов и все маршруты.


In [ ]:
def blend(frame, weight):
    return weight * frame.catboost + (1 - weight) * frame.seasonal

tuning = validation.loc[validation.cutoff < holdout_start]
holdout = validation.loc[validation.cutoff >= holdout_start]
candidates = []
for w in np.linspace(0, 1, 21):
    mixed = blend(tuning, w)
    for k in (1.0, tuning.y.sum() / mixed.sum()):
        candidates.append((mean_daily_score(tuning, mixed * k), w, k))
_, weight, factor = min(candidates)
print(f"Вес CatBoost: {weight:.2f}, множитель: {factor:.4f}")
for label, prediction in [("Сезонный", holdout.seasonal), ("CatBoost", holdout.catboost),
                          ("Смесь", blend(holdout, weight)),
                          ("Смесь с калибровкой", blend(holdout, weight) * factor)]:
    print(f"{label}: {mean_daily_score(holdout, prediction):.5f}")

## Три эксперимента
Сравниваем две модели h=2 в исходных единицах. Для первого горизонта проверяем замену всей исходной смеси прогнозом из перекрытия. Для остальных пробуем добавить нормализованный профиль слота за 28 дней. Вес профиля и состав итогового прогноза выбираем по первым 14 проверочным датам, а последние 14 дней и субботы показываем отдельно. На ранних стартах профиль рассчитывается по доступным прошлым дням.


In [ ]:
def experiment_prediction(frame, overlap, scaled, profile_weight):
    base = blend(frame, weight) * factor
    if scaled:
        second = frame.horizon.eq(2)
        base.loc[second] = (weight * frame.loc[second, "scaled"] +
                            (1 - weight) * frame.loc[second, "seasonal"]) * factor
    if overlap:
        first = frame.horizon.eq(1)
        base.loc[first] = frame.loc[first, "overlap"]
    later = frame.horizon.gt(1)
    base.loc[later] = (1 - profile_weight) * base.loc[later] + profile_weight * frame.loc[later, "profile28"]
    return base

second = holdout.loc[holdout.horizon.eq(2)]
display(pd.DataFrame([{"модель": name, "MAE": (second[name] - second.y).abs().mean(),
                       "std прогноза": second[name].std(), "метрика h2": mean_daily_score(second, second[name])}
                      for name in ("catboost", "scaled")]))
choices = [(overlap, scaled, float(w)) for overlap in (False, True)
           for scaled in (False, True) for w in np.linspace(0, 1, 11)]
def tune_score(choice):
    return mean_daily_score(tuning, experiment_prediction(tuning, *choice))

chosen = min(choices, key=tune_score)
profile_only = min([c for c in choices if not c[0] and not c[1]], key=tune_score)
rows = []
for label, choice in [("исходная смесь", (False, False, 0)), ("только замена h1", (True, False, 0)),
                      ("только замена модели h2", (False, True, 0)), ("только профиль", profile_only),
                      ("выбранная комбинация", chosen)]:
    row = {"вариант": label, "вес профиля": choice[2], "подбор": tune_score(choice)}
    for period, subset in [("проверка", holdout), ("субботы", holdout.loc[holdout.cutoff.dt.dayofweek.eq(5)])]:
        row[period] = mean_daily_score(subset, experiment_prediction(subset, *choice))
    rows.append(row)
display(pd.DataFrame(rows))
print("Замена h1, масштабирование h2, вес профиля:", chosen)


In [ ]:
early = pd.read_parquet(OUT / "history_h1.parquet")
early = early.loc[early.cutoff.isin(dates[:-28])].copy()
early["seasonal"] = early[["slot_daily7", "slot_weekly4"]].mean(axis=1)
early = early.dropna(subset=["seasonal", "overlap"])
print("Ранняя история, метрика отдельно для h1:")
for name in ("seasonal", "overlap"):
    print(name, mean_daily_score(early, early[name]))
del early

## Финальное обучение на всей размеченной истории
После выбора настроек переобучаем все варианты с сохранённым числом деревьев. После каждого обучения показываем оценку на всей истории и сохранённые train/test предыдущей модели. Даты локального test теперь включены в обучение, поэтому независимая оценка относится к предыдущему этапу.


In [ ]:
submission_parts = []
for h in range(1, 9):
    data = pd.read_parquet(OUT / f"history_h{h}.parquet")
    future = pd.read_parquet(OUT / f"submission_h{h}.parquet")
    columns = model_columns(data)
    result = future[["id", "profile28", "overlap"]].copy()
    result["seasonal"] = future[["slot_daily7", "slot_weekly4"]].mean(axis=1)
    result["horizon"] = h
    for variant, scale in variants(h):
        model = make_model(tree_counts[h, variant])
        model.fit(data[columns], data.y / scale)
        suffix = "_scaled" if variant == "scaled" else ""
        model.save_model(str(MODEL_DIR / f"catboost_h{h}{suffix}.cbm"))
        result[variant] = (model.predict(future[columns]) * scale).clip(min=0)
        full_pred = (model.predict(data[columns]) * scale).clip(min=0)
        full_row = score_row(data, full_pred, "вся история после переобучения")
        print(f"h={h}, {variant}: первые две строки — оценка до переобучения")
        display(pd.concat([evaluation_tables[h, variant], pd.DataFrame([full_row])], ignore_index=True))
        del model
    submission_parts.append(result)

## Итоговый файл для соревнования


In [ ]:
predictions = pd.concat(submission_parts, ignore_index=True)
predictions["y_pred"] = experiment_prediction(predictions, *chosen)
submission = test[["id"]].merge(predictions[["id", "y_pred"]], on="id", how="left",
                                validate="one_to_one")
if len(predictions) != len(test) or not np.isfinite(submission.y_pred).all():
    raise ValueError("Неполный прогноз для тестовой выборки")
submission.to_csv("submission.csv", index=False)
display(submission.head())